# Notebook 08 — WinIT (Window-based Importance for Time series)

**Reference**: Leung et al., *WinIT: Explaining Time Series Predictions via Window-based Importance Scores*, ICLR 2023.

**Dataset**: Robot diagnostic data (81 450 rows, 62 active sensors, 4 classes).

**Idea**: For each timestep *t* in a sequence, compute how much the suffix starting at *t* contributes to the final prediction by replacing that suffix with a baseline (training-set mean). The marginal contribution of timestep *t* is the finite difference of successive suffix-masked predictions.

$$\text{importance}(t) = \bigl[f(x) - f(\tilde{x}_{t:T})\bigr] - \bigl[f(x) - f(\tilde{x}_{t+1:T})\bigr] = f(\tilde{x}_{t+1:T}) - f(\tilde{x}_{t:T})$$

where $\tilde{x}_{t:T}$ is the sequence with timesteps $[t, T)$ replaced by the baseline.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

print('imports ok')

## 1 — Load and preprocess robot data

In [ ]:
DATA_PATH = pathlib.Path('../data/_RobotData/labeled_data/diagnostic model.xlsx')
df = pd.read_excel(DATA_PATH)
print(f'Loaded {df.shape[0]:,} rows × {df.shape[1]} cols')
print('Class distribution:')
print(df['Outcome'].value_counts().sort_index().rename({0:'Normal',1:'Failure-1',2:'Failure-2',3:'Failure-3'}))

In [ ]:
# Drop label + constant columns
meta_cols  = ['Outcome']
drop_const = [c for c in df.columns if c not in meta_cols and df[c].nunique() <= 1]
feature_cols = [c for c in df.columns if c not in meta_cols + drop_const]
print(f'Active features: {len(feature_cols)}  (dropped {len(drop_const)} constant cols)')

X_raw = df[feature_cols].values.astype(np.float32)
y_raw = df['Outcome'].values.astype(np.int64)

N_CLASSES = 4
CLASS_NAMES = ['Normal', 'Failure-1', 'Failure-2', 'Failure-3']

In [ ]:
from sklearn.preprocessing import MinMaxScaler

WINDOW = 30
STEP   = 15
SPLIT  = 0.8

split_idx = int(len(X_raw) * SPLIT)
scaler = MinMaxScaler()
X_train_raw = scaler.fit_transform(X_raw[:split_idx])
X_test_raw  = scaler.transform(X_raw[split_idx:])

def make_windows(X, y, window, step):
    Xs, ys = [], []
    for i in range(0, len(X) - window, step):
        Xs.append(X[i:i+window])
        ys.append(y[i+window-1])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.int64)

X_train, y_train = make_windows(X_train_raw, y_raw[:split_idx],  WINDOW, STEP)
X_test,  y_test  = make_windows(X_test_raw,  y_raw[split_idx:],  WINDOW, STEP)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print('Train class counts:', dict(zip(*np.unique(y_train, return_counts=True))))
print('Test  class counts:', dict(zip(*np.unique(y_test,  return_counts=True))))

## 2 — Train LSTM with class weights

In [ ]:
from collections import Counter
from src.models.lstm_classifier import LSTMTrainer

counts = Counter(y_train.tolist())
total  = sum(counts.values())
weights = torch.tensor(
    [total / (N_CLASSES * counts[c]) for c in range(N_CLASSES)],
    dtype=torch.float32,
)
print('Class weights:', {CLASS_NAMES[i]: round(weights[i].item(), 3) for i in range(N_CLASSES)})

lstm = LSTMTrainer(
    input_size=len(feature_cols), hidden_size=128, num_layers=2,
    num_classes=N_CLASSES, dropout=0.3, lr=1e-3, epochs=40,
    batch_size=256, class_weights=weights,
)
lstm.fit(X_train, y_train)
print('Training complete.')

In [ ]:
from sklearn.metrics import classification_report

y_pred = lstm.predict(X_test)
print(classification_report(
    y_test, y_pred,
    labels=[0,1,2,3],
    target_names=CLASS_NAMES,
    zero_division=0,
))

## 3 — WinIT implementation (from scratch)

In [ ]:
# Baseline: per-feature mean of the TRAINING set (shape: (1, WINDOW, n_features))
# We broadcast this across all timesteps
baseline_feat = X_train.mean(axis=0)  # (WINDOW, n_features) — mean window
# Use a flat mean so each feature gets its global mean regardless of timestep
baseline_flat = X_train.mean(axis=(0, 1))  # (n_features,)
baseline_row  = np.tile(baseline_flat, (WINDOW, 1)).astype(np.float32)  # (WINDOW, n_features)

print(f'Baseline shape: {baseline_row.shape}  (mean of training set per feature)')

In [ ]:
def winit_importance_single(model_fn, X_sample, baseline):
    """
    Compute WinIT temporal importance for one sample.

    For each timestep t:
      raw[t]  = f(x) - f(x with [t:T] replaced by baseline)
      raw[T]  = 0  (nothing masked)
      importance[t] = raw[t] - raw[t+1]  (marginal contribution of timestep t)

    Returns: importance array (T, C)
    """
    T, F = X_sample.shape
    pred_orig = model_fn(X_sample[None])[0]       # (C,)
    n_classes = len(pred_orig)

    # raw[T] = 0 by definition
    raw = np.zeros((T + 1, n_classes), dtype=np.float64)

    for t in range(T):
        x_masked = X_sample.copy()
        x_masked[t:] = baseline[t:]              # mask suffix [t, T)
        pred_masked = model_fn(x_masked[None])[0] # (C,)
        raw[t] = pred_orig - pred_masked
    # raw[T] stays 0

    importance = raw[:T] - raw[1:T+1]             # (T, C)
    return importance


def model_fn(X_3d):
    """Wrapper: (n, T, F) → (n, C) probabilities."""
    return lstm.predict_proba(X_3d)


print('WinIT function defined.')

## 4 — Run WinIT on representative samples per class

In [ ]:
N_EXPLAIN = 20   # samples per class to explain (WinIT is slower than KernelSHAP per-sample)

# Build explain set: N_EXPLAIN samples per class
# If test set lacks a class, fall back to train
explain_X = []
explain_y = []

rng = np.random.default_rng(42)
for cls in range(N_CLASSES):
    test_idx  = np.where(y_test == cls)[0]
    train_idx = np.where(y_train == cls)[0]

    if len(test_idx) >= N_EXPLAIN:
        chosen = rng.choice(test_idx, N_EXPLAIN, replace=False)
        explain_X.append(X_test[chosen])
        src = 'test'
    elif len(test_idx) > 0:
        chosen = rng.choice(test_idx, N_EXPLAIN, replace=True)
        explain_X.append(X_test[chosen])
        src = 'test (oversampled)'
    else:
        chosen = rng.choice(train_idx, N_EXPLAIN, replace=False)
        explain_X.append(X_train[chosen])
        src = 'train (test lacks this class)'

    explain_y.extend([cls] * N_EXPLAIN)
    print(f'  Class {cls} ({CLASS_NAMES[cls]}): {N_EXPLAIN} samples from {src}')

explain_X = np.concatenate(explain_X, axis=0)  # (N_CLASSES*N_EXPLAIN, WINDOW, n_features)
explain_y = np.array(explain_y)
print(f'\nExplain set shape: {explain_X.shape}')

In [ ]:
from tqdm.auto import tqdm

# winit_scores[i] → (WINDOW, N_CLASSES) importance for sample i
winit_scores = []

for i in tqdm(range(len(explain_X)), desc='WinIT'):
    imp = winit_importance_single(model_fn, explain_X[i], baseline_row)
    winit_scores.append(imp)

winit_scores = np.array(winit_scores)  # (N_TOTAL, WINDOW, N_CLASSES)
print(f'WinIT scores shape: {winit_scores.shape}  — (n_samples, timesteps, n_classes)')

## 5 — Temporal importance: which timesteps matter most per class?

In [ ]:
# For each true class, average the importance of the matching class channel
# mean_temporal[cls] → (WINDOW,) mean importance of timestep t for class cls
mean_temporal = {}
for cls in range(N_CLASSES):
    mask  = (explain_y == cls)
    # take absolute value — both positive & negative shifts matter
    mean_temporal[cls] = np.abs(winit_scores[mask, :, cls]).mean(axis=0)  # (WINDOW,)

# Plot
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharey=False)
colors = ['#4c72b0','#dd8452','#55a868','#c44e52']

for cls, ax in enumerate(axes.flat):
    imp = mean_temporal[cls]
    ax.bar(range(WINDOW), imp, color=colors[cls], alpha=0.85)
    ax.set_title(f'{CLASS_NAMES[cls]}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Timestep (0 = oldest, 29 = most recent)')
    ax.set_ylabel('Mean |WinIT importance|')
    ax.axvline(WINDOW - 1, color='black', linestyle='--', alpha=0.4, label='last step')
    ax.xaxis.set_major_locator(mticker.MultipleLocator(5))

plt.suptitle('WinIT Temporal Importance per Class (Robot Data)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/winit_temporal_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/winit_temporal_importance.png')

## 6 — Feature importance at the most important timestep per class

In [ ]:
def winit_feature_importance_single(model_fn, X_sample, baseline, timestep):
    """
    For a specific timestep already identified as important,
    compute per-feature importance via leave-one-feature-out at that timestep.

    Masks suffix [timestep:T] then unmasks one feature at a time to measure
    how much each feature contributes at that timestep.

    Returns: feat_imp (n_features, C)
    """
    T, F = X_sample.shape

    # Prediction with suffix fully masked (reference)
    x_ref = X_sample.copy()
    x_ref[timestep:] = baseline[timestep:]
    pred_ref = model_fn(x_ref[None])[0]   # (C,)

    # Prediction with suffix masked except feature f at timestep
    feat_imp = np.zeros((F, len(pred_ref)), dtype=np.float64)
    for f in range(F):
        x_f = x_ref.copy()
        x_f[timestep, f] = X_sample[timestep, f]   # restore feature f
        pred_f = model_fn(x_f[None])[0]
        feat_imp[f] = pred_f - pred_ref             # contribution of feature f

    return feat_imp  # (F, C)

print('Feature importance function defined.')

In [ ]:
TOP_T = 3   # analyse the top-3 most important timesteps per class
TOP_F = 15  # show top-15 features

feat_imp_per_class = {}   # cls → (F,) mean |feature importance|

for cls in range(N_CLASSES):
    mask   = (explain_y == cls)
    X_cls  = explain_X[mask]           # (N_EXPLAIN, WINDOW, F)
    scores = winit_scores[mask]         # (N_EXPLAIN, WINDOW, C)

    # Top-T timesteps (by mean temporal importance for this class)
    top_t = np.argsort(mean_temporal[cls])[-TOP_T:]

    all_feat_imp = []
    for i in range(len(X_cls)):
        for t in top_t:
            fi = winit_feature_importance_single(model_fn, X_cls[i], baseline_row, int(t))
            all_feat_imp.append(np.abs(fi[:, cls]))  # (F,)

    feat_imp_per_class[cls] = np.mean(all_feat_imp, axis=0)  # (F,)
    top_features = np.argsort(feat_imp_per_class[cls])[-5:][::-1]
    print(f'{CLASS_NAMES[cls]} top-5 features: {[feature_cols[f] for f in top_features]}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for cls, ax in enumerate(axes.flat):
    fi   = feat_imp_per_class[cls]                       # (F,)
    idxs = np.argsort(fi)[-TOP_F:][::-1]                # top-F descending
    vals = fi[idxs]
    names = [feature_cols[i] for i in idxs]

    ax.barh(range(len(names)), vals, color=colors[cls], alpha=0.85)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(f'{CLASS_NAMES[cls]} — feature importance at key timesteps', fontsize=11, fontweight='bold')
    ax.set_xlabel('Mean |WinIT feature contribution|')

plt.suptitle('WinIT Feature Importance per Class (Robot Data)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/winit_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/winit_feature_importance.png')

## 7 — Summary heatmap: temporal × feature importance

In [ ]:
# For each class, build a (WINDOW, TOP_F) heatmap showing which feature matters at which timestep
def compute_temporal_feature_map(model_fn, X_cls, baseline_row, cls, top_f_idxs):
    """
    Returns (WINDOW, len(top_f_idxs)) mean |importance| matrix.
    At each timestep t, do feature-level attribution.
    """
    T = X_cls.shape[1]
    F_sub = len(top_f_idxs)
    matrix = np.zeros((T, F_sub))

    for i in range(len(X_cls)):
        for t in range(T):
            fi = winit_feature_importance_single(model_fn, X_cls[i], baseline_row, t)
            matrix[t] += np.abs(fi[top_f_idxs, cls])

    return matrix / len(X_cls)


TOP_F_HEAT = 10   # top features per class for the heatmap
N_HEAT     = 5    # samples per class (heatmap is expensive: T × N)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

for cls, ax in enumerate(axes.flat):
    mask     = (explain_y == cls)
    X_cls    = explain_X[mask][:N_HEAT]

    top_f_idxs = np.argsort(feat_imp_per_class[cls])[-TOP_F_HEAT:][::-1]
    feat_names = [feature_cols[i] for i in top_f_idxs]

    print(f'Computing heatmap for {CLASS_NAMES[cls]} ...')
    heatmap = compute_temporal_feature_map(model_fn, X_cls, baseline_row, cls, top_f_idxs)
    # heatmap: (WINDOW, TOP_F_HEAT)

    im = ax.imshow(heatmap.T, aspect='auto', cmap='YlOrRd', origin='upper')
    ax.set_yticks(range(TOP_F_HEAT))
    ax.set_yticklabels(feat_names, fontsize=8)
    ax.set_xlabel('Timestep →')
    ax.set_title(f'{CLASS_NAMES[cls]}', fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, label='|importance|')

plt.suptitle('WinIT Temporal × Feature Heatmap (Robot Data)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/winit_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/winit_heatmap.png')

## 8 — Compare top sensors: WinIT vs KernelSHAP

In [ ]:
TOP_N = 10

print('=' * 65)
print(f'WinIT vs KernelSHAP — top-{TOP_N} features per class')
print('=' * 65)

for cls in range(N_CLASSES):
    fi       = feat_imp_per_class[cls]
    top_idxs = np.argsort(fi)[-TOP_N:][::-1]
    top_feats = [feature_cols[i] for i in top_idxs]

    print(f'\n{CLASS_NAMES[cls]}:')
    print(f'  WinIT top-{TOP_N}:   {top_feats}')
    # KernelSHAP results from notebook 07 (from class-weighted model)
    # Update these after re-running notebook 07
    print(f'  KernelSHAP top-{TOP_N}: [run notebook 07 with class weights to populate]')

print('\nNote: KernelSHAP flattens (30,62)→1860 and uses OLS; WinIT works on the')
print('native (30,62) sequence and captures when each feature matters.')

## 9 — Save WinIT results

In [ ]:
out_dir = pathlib.Path('../results')
out_dir.mkdir(exist_ok=True)

np.save(out_dir / 'winit_scores.npy',    winit_scores)    # (N_TOTAL, WINDOW, N_CLASSES)
np.save(out_dir / 'winit_explain_y.npy', explain_y)

# Feature importance per class
for cls in range(N_CLASSES):
    np.save(out_dir / f'winit_feat_imp_cls{cls}.npy', feat_imp_per_class[cls])

print('Saved WinIT scores and feature importances to results/')
print(f'  winit_scores.npy:       {winit_scores.shape}')
print(f'  winit_feat_imp_cls*.npy: {feat_imp_per_class[0].shape} per class')